In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate pandas tqdm

In [ ]:
import torch
import gc
import os
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from tqdm import tqdm
import torch.nn as nn
from transformers import AutoModel

# uso la mia classe personalizzata usata durante il training
class DebertaJudge(nn.Module):
    def __init__(self, model_name="microsoft/deberta-v3-base"):
        super().__init__()
        # Forziamo il modello in float32 puro per evitare ogni problema di precisione
        self.bert = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        self.bert.config.use_cache = False
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)
        
    def forward(self, input_ids, mask):
        out = self.bert(input_ids, attention_mask=mask)
        return self.regressor(out.last_hidden_state[:, 0, :])

def calculate_perplexity(texts, model_id="EleutherAI/pythia-6.9b"):
    print(f"\n Caricamento Modello Giudice PPL: {model_id}")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token 
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config, device_map="auto"
    )
    model.eval()

    total_loss, total_length = 0.0, 0
    with torch.no_grad():
        for text in tqdm(texts, desc="Calcolo Perplexity (PPL)"):
            encodings = tokenizer(text, return_tensors="pt").to("cuda")
            seq_len = encodings.input_ids.size(1)
            if seq_len < 2: continue
            outputs = model(encodings.input_ids, labels=encodings.input_ids)
            total_loss += outputs.loss.item() * seq_len
            total_length += seq_len

    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    return torch.exp(torch.tensor(total_loss / total_length)).item() if total_length > 0 else 0.0

def calculate_reward_mean(texts, reward_model_path, base_model_name="microsoft/deberta-v3-base"):
    print(f"\n Caricamento Modello Reward Custom dal file .pth: {reward_model_path}")
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    
    # Inizializziamo l'esatta classe usata nel training
    model = DebertaJudge()
    
    # Ora i pesi entreranno come un guanto
    state_dict = torch.load(reward_model_path, map_location="cuda")
    model.load_state_dict(state_dict)
    
    model.to("cuda")
    model.eval()

    total_reward = 0.0
    with torch.no_grad():
        for text in tqdm(texts, desc="Calcolo Reward Mean"):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
            
            # ATTENZIONE QUI: Passiamo 'mask' invece di 'attention_mask' perché la tua classe vuole così!
            reward = model(input_ids=inputs['input_ids'], mask=inputs['attention_mask'])
            total_reward += reward.squeeze().item()

    # Pulizia memoria profonda
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    
    return total_reward / len(texts) if len(texts) > 0 else 0.0

# 1. Carica il dataset FINALE 
df = pd.read_csv("/kaggle/input/datasets/silvioliparoti/dataset-ultimo/dataset_valutazione_finale.csv")

# 2. Definisci i path dei modelli giudici
MODELLO_BASE_PPL = "EleutherAI/pythia-6.9b" 
PATH_DEBERTA = "/kaggle/input/datasets/silvioliparoti/deberta-finale/deberta_judge_v2_2_FINAL.pth"

# 3. Creazione del file di salvataggio (Aggiunta la colonna Categoria)
FILE_RISULTATI = "risultati_metriche.csv"
if not os.path.exists(FILE_RISULTATI):
    pd.DataFrame(columns=["Modello", "Categoria", "Perplexity (PPL)", "Reward Mean"]).to_csv(FILE_RISULTATI, index=False)

# Funzioncina comoda per salvare (Modificata per accettare la Categoria)
def salva_metrica(nome, categoria, ppl, reward):
    df_res = pd.read_csv(FILE_RISULTATI)
    # Rimuove la riga vecchia se stiamo ricalcolando la stessa categoria per lo stesso modello
    df_res = df_res[~((df_res["Modello"] == nome) & (df_res["Categoria"] == categoria))] 
    nuova_riga = pd.DataFrame([{"Modello": nome, "Categoria": categoria, "Perplexity (PPL)": round(ppl, 3), "Reward Mean": round(reward, 3)}])
    df_res = pd.concat([df_res, nuova_riga], ignore_index=True)
    df_res.to_csv(FILE_RISULTATI, index=False)
    print(f" Salvato in {FILE_RISULTATI}!")

print(" Setup completato")

In [ ]:
nome_modello = "Zephyr_Base"
print(f" Valutazione in corso per: {nome_modello}")

# Divido le battute tramite la colonna Usa_RAG
battute_all = df[nome_modello].dropna().tolist()
battute_pure = df[df["Usa_RAG"] == False][nome_modello].dropna().tolist()
battute_rag = df[df["Usa_RAG"] == True][nome_modello].dropna().tolist()

print(f"\n--- 1. Calcolo PURE (No RAG) ---")
ppl_pure = calculate_perplexity(battute_pure, model_id=MODELLO_BASE_PPL)
reward_pure = calculate_reward_mean(battute_pure, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "Pure", ppl_pure, reward_pure)

print(f"\n--- 2. Calcolo ATTUALITÀ (Con RAG) ---")
ppl_rag = calculate_perplexity(battute_rag, model_id=MODELLO_BASE_PPL)
reward_rag = calculate_reward_mean(battute_rag, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "RAG", ppl_rag, reward_rag)

print(f"\n--- 3. Calcolo COMPLESSIVO ---")
ppl_all = calculate_perplexity(battute_all, model_id=MODELLO_BASE_PPL)
reward_all = calculate_reward_mean(battute_all, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "Complessivo", ppl_all, reward_all)

print(f"\n RISULTATI FINALI {nome_modello}: PPL={ppl_all:.2f} | Reward={reward_all:.2f}")

In [ ]:
nome_modello = "PPO_v2"
print(f" Valutazione in corso per: {nome_modello}")

battute_all = df[nome_modello].dropna().tolist()
battute_pure = df[df["Usa_RAG"] == False][nome_modello].dropna().tolist()
battute_rag = df[df["Usa_RAG"] == True][nome_modello].dropna().tolist()

print(f"\n--- 1. Calcolo PURE (No RAG) ---")
ppl_pure = calculate_perplexity(battute_pure, model_id=MODELLO_BASE_PPL)
reward_pure = calculate_reward_mean(battute_pure, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "Pure", ppl_pure, reward_pure)

print(f"\n--- 2. Calcolo ATTUALITÀ (Con RAG) ---")
ppl_rag = calculate_perplexity(battute_rag, model_id=MODELLO_BASE_PPL)
reward_rag = calculate_reward_mean(battute_rag, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "RAG", ppl_rag, reward_rag)

print(f"\n--- 3. Calcolo COMPLESSIVO ---")
ppl_all = calculate_perplexity(battute_all, model_id=MODELLO_BASE_PPL)
reward_all = calculate_reward_mean(battute_all, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "Complessivo", ppl_all, reward_all)

print(f"\n RISULTATI FINALI {nome_modello}: PPL={ppl_all:.2f} | Reward={reward_all:.2f}")

In [ ]:
nome_modello = "DPO_v2.2"
print(f" Valutazione in corso per: {nome_modello}")

battute_all = df[nome_modello].dropna().tolist()
battute_pure = df[df["Usa_RAG"] == False][nome_modello].dropna().tolist()
battute_rag = df[df["Usa_RAG"] == True][nome_modello].dropna().tolist()

print(f"\n--- 1. Calcolo PURE (No RAG) ---")
ppl_pure = calculate_perplexity(battute_pure, model_id=MODELLO_BASE_PPL)
reward_pure = calculate_reward_mean(battute_pure, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "Pure", ppl_pure, reward_pure)

print(f"\n--- 2. Calcolo ATTUALITÀ (Con RAG) ---")
ppl_rag = calculate_perplexity(battute_rag, model_id=MODELLO_BASE_PPL)
reward_rag = calculate_reward_mean(battute_rag, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "RAG", ppl_rag, reward_rag)

print(f"\n--- 3. Calcolo COMPLESSIVO ---")
ppl_all = calculate_perplexity(battute_all, model_id=MODELLO_BASE_PPL)
reward_all = calculate_reward_mean(battute_all, reward_model_path=PATH_DEBERTA)
salva_metrica(nome_modello, "Complessivo", ppl_all, reward_all)

print(f"\n RISULTATI FINALI {nome_modello}: PPL={ppl_all:.2f} | Reward={reward_all:.2f}")